# Modelos e Avaliação

Especificação, estimação e avaliação out-of-sample dos modelos competidores.

| Modelo | Especificação |
|--------|---------------|
| M0 — Random Walk | σ²ₜ = RV_{t-1} |
| M1 — GARCH(1,1) | σ²ₜ = ω + α·ε²_{t-1} + β·σ²_{t-1} |
| M2 — GARCH-X(1,1) | σ²ₜ = ω + α·ε²_{t-1} + β·σ²_{t-1} + γ·ΔSVI_{t-1} |

Requer os CSVs gerados por `01_coleta_dados.ipynb` na pasta `./data/`.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from arch import arch_model
from arch.bootstrap import MCS

CAMINHO = "./data/"

# Bases salvas

In [2]:
df_acoes = pd.read_csv("./data/acoes_elegiveis.csv")
df_trends = pd.read_csv("./data/google_trends_svi.csv", parse_dates=["semana"])
df_precos = pd.read_csv("./data/precos_semanais.csv", parse_dates=["semana"])
df_log = pd.read_csv("./data/log_coleta_svi.csv")
df_adf = pd.read_csv("./data/log_adf_svi.csv")
df_final = pd.read_csv("./data/base_final_tcc.csv", parse_dates=["semana"])

df_previsoes = pd.read_csv("./data/previsoes_consolidadas.csv", parse_dates=["semana"])
df_resumo_params = pd.read_csv("./data/resumo_parametros.csv")

# Especificação dos Modelos Competidores

## MODELO 0: PASSEIO ALEATÓRIO (Random Walk)
σ̂²_t = RV_{t-1}

In [6]:
def passeio_aleatorio(grupo):
    df_t = (
        grupo.sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv"]]
        .copy()
    )
    df_t["rv_pred_m0"] = df_t["rv"].shift(1)
    return df_t.dropna(subset=["rv_pred_m0"])

## MODELO 1: GARCH(1,1) PADRÃO
σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1}

Janela móvel de tamanho fixo: 52 * 3 semanas

In [7]:
def garch_padrao(grupo, col_end="ret_semanal", janela=52 * 3):
    df_t = (
        grupo.sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv", col_end]]
        .copy()
    )
    n, previsoes = len(df_t), []

    for t in range(janela, n):
        treino = df_t[col_end].iloc[t - janela : t].values.astype(float) * 100
        try:
            fit = arch_model(treino, mean="Constant", vol="GARCH",
                             p=1, q=1, dist="normal").fit(disp="off", show_warning=False)
            var_pred = fit.forecast(horizon=1, reindex=False).variance.values[-1, 0] / 10_000
            previsoes.append({
                "semana":     df_t["semana"].iloc[t],
                "rv":          df_t["rv"].iloc[t],
                col_end:         df_t[col_end].iloc[t],
                "rv_pred_m1": var_pred,
                "omega":      fit.params.get("omega",    np.nan),
                "alpha":      fit.params.get("alpha[1]", np.nan),
                "beta":       fit.params.get("beta[1]",  np.nan),
            })
        except Exception as e:
            print(f"  [ERRO GARCH t={t}]: {e}")
            previsoes.append({"semana": df_t["semana"].iloc[t], "rv": df_t["rv"].iloc[t], col_end: df_t[col_end].iloc[t],
                              "rv_pred_m1": np.nan, "omega": np.nan, "alpha": np.nan, "beta": np.nan})

    return pd.DataFrame(previsoes)

## MODELO 2: GARCH-X(1,1)
$(σ²_t = ω + α·ε²_{t-1} + β·σ²_{t-1} + γ·ΔSVI_{t-1})$

Janela móvel de tamanho fixo: 52 * 3 semanas

---
O que muda em relação ao Modelo 1:

A única diferença estrutural é o exógeno $\gamma \cdot \Delta SVI_{t-1}$, que aparece em dois momentos distintos:

**Na estimação** — `x=svi_treino` passa os 156 valores de $\Delta SVI$ da janela para a biblioteca `arch` estimar o parâmetro $\gamma$ junto com $(\omega, \alpha, \beta)$.

**Na previsão** — `x=svi_previsao` passa apenas o valor $\Delta SVI_{t-1}$, que é o único ponto do exógeno disponível no momento em que a previsão é feita. Isso garante que não há *look-ahead bias* — na semana $t$ você só conhece o SVI até a semana $t-1$.

In [8]:
def garch_exog(grupo, col_end="ret_semanal", col_exog="svi_diff", janela=52 * 3):
    df_t = (
        grupo.sort_values("semana")
        .reset_index(drop=True)
        [["semana", "rv", col_end, col_exog]]
        .dropna(subset=[col_exog])
        .reset_index(drop=True)
        .copy()
    )
    n, previsoes = len(df_t), []

    if n <= janela:
        print(f"  [AVISO] Obs. insuficientes ({n}), retornando vazio.")
        return pd.DataFrame()

    for t in range(janela, n):
        treino        = df_t[col_end].iloc[t - janela : t].values.astype(float)  * 100
        exog_treino   = df_t[col_exog].iloc[t - janela : t].values.astype(float).reshape(-1, 1)
        exog_forecast = df_t[col_exog].iloc[t : t + 1].values.astype(float).reshape(1, 1)
        try:
            fit = arch_model(treino, x=exog_treino, mean="ARX", vol="GARCH",
                             p=1, q=1, dist="normal").fit(disp="off", show_warning=False)
            var_pred = fit.forecast(horizon=1, reindex=False, x=exog_forecast).variance.values[-1, 0] / 10_000

            gamma_key  = next((k for k in fit.params.index  if k.startswith("x")), None)
            gamma      = fit.params[gamma_key]  if gamma_key else np.nan
            gamma_pval = fit.pvalues[gamma_key] if gamma_key else np.nan

            previsoes.append({
                "semana":      df_t["semana"].iloc[t],
                "rv":          df_t["rv"].iloc[t],
                col_end:       df_t[col_end].iloc[t],
                "rv_pred_m2":  var_pred,
                "omega":       fit.params.get("omega",    np.nan),
                "alpha":       fit.params.get("alpha[1]", np.nan),
                "beta":        fit.params.get("beta[1]",  np.nan),
                "gamma":       gamma,
                "gamma_pval":  gamma_pval,
                "status_m2":      "ok",
            })
        except Exception as e:
            print(f"  [ERRO GARCH-X t={t}]: {e}")
            previsoes.append({"semana": df_t["semana"].iloc[t], "rv": df_t["rv"].iloc[t], col_end: df_t[col_end].iloc[t],
                              "rv_pred_m2": np.nan, "omega": np.nan, "alpha": np.nan,
                              "beta": np.nan, "gamma": np.nan, "gamma_pval": np.nan, "status_m2": str(e),})

    return pd.DataFrame(previsoes)

# Avaliação de Previsão e Inferência (Modelos Aninhados)
```
Janela de treino:   svi_diff[ t-156 … t-1 ]  →  estima γ
Previsão:           svi_diff[ t-1 ]           →  usa γ para prever σ²_t



In [11]:
# ── Proporção de zeros no SVI por ticker ─────────────────────────────
# Calculada aqui para uso nos três cenários de avaliação.
# O filtro NÃO é aplicado sobre df_final — todos os tickers entram
# na modelagem; o corte é feito na etapa de avaliação (texto principal
# e apêndice), permitindo comparação direta entre cenários.
# IMPORTANTE: re-execute a célula do Loop Principal após esta alteração
# para regenerar df_previsoes com todos os tickers.
zeros_pct = df_trends.groupby("ticker_b3")["svi"].apply(lambda x: (x == 0).mean())

In [12]:
# ── Configurações ─────────────────────────────────────────────────
JANELA = 52 * 3

NOMES = {
    "qlike_m0": "M0 (RW)",
    "qlike_m1": "M1 (GARCH)",
    "qlike_m2": "M2 (GARCH-X)",
}

In [ ]:
# ── Loop principal ─────────────────────────────────────────────────
resultados, resumo_params = [], []

for ticker, grupo in df_final.groupby("ticker_b3"):
    print(f"► {ticker}", end=" ... ")

    df_t = grupo.sort_values("semana").reset_index(drop=True)

    if len(df_t) <= JANELA:
      print("insuficiente, pulando.")
      continue

    semana_corte = df_t["semana"].iloc[JANELA]

    df_m0 = passeio_aleatorio(grupo)
    df_m1 = garch_padrao(grupo, janela=JANELA)
    df_m2 = garch_exog(grupo, janela=JANELA)

    if df_m2.empty:
        continue

    # ── Merge ──────────────────────────────────────────────────────
    # rv_pred_* = previsão de variância condicional (saída do GARCH)
    # rv        = volatilidade realizada (benchmark para QLIKE)
    df_tick = (
        df_m1
        .merge(df_m0[["semana", "rv_pred_m0"]], on="semana", how="left")
        .merge(df_m2[["semana", "rv_pred_m2", "gamma", "gamma_pval", "status_m2"]], on="semana", how="left")
    )
    df_tick["ticker_b3"] = ticker
    df_tick["split"]     = np.where(
        df_tick["semana"] >= semana_corte, "out-of-sample", "in-sample"
    )
    resultados.append(df_tick)

    # ── Resumo parâmetros M2 ───────────────────────────────────────
    resumo_params.append({
        "ticker_b3":        ticker,
        "n_previsoes":      len(df_m2),
        "gamma_medio":      df_m2["gamma"].mean(),
        "gamma_pval_medio": df_m2["gamma_pval"].mean(),
        "pct_signif_5pct":  (df_m2["gamma_pval"] < 0.05).mean() * 100,
        "persistencia_m1":  (df_m1["alpha"] + df_m1["beta"]).mean(),
        "persistencia_m2":  (df_m2["alpha"] + df_m2["beta"]).mean(),
    })
    print("✓")

# ── Consolida ──────────────────────────────────────────────────────
df_previsoes     = pd.concat(resultados, ignore_index=True)
df_resumo_params = pd.DataFrame(resumo_params).set_index("ticker_b3")

# ── Salva ──────────────────────────────────────────────────────────
df_previsoes.to_csv(CAMINHO + "previsoes_consolidadas.csv", index=False)
df_resumo_params.to_csv(CAMINHO + "resumo_parametros.csv")

In [13]:
# Preserva versão completa (usada na comparação diagnóstica)
ticker_erro_m2 = df_previsoes[df_previsoes.status_m2 == "Singular matrix"]["ticker_b3"].unique()
df_previsoes_raw = df_previsoes.copy()  # inclui tickers com erro
df_previsoes      = df_previsoes[~df_previsoes.ticker_b3.isin(ticker_erro_m2)]  # sem erros

print(f"Tickers em df_previsoes_raw : {df_previsoes_raw['ticker_b3'].nunique()}")
print(f"Tickers com singular matrix : {len(ticker_erro_m2)}")
print(f"Tickers em df_previsoes     : {df_previsoes['ticker_b3'].nunique()}")

Tickers em df_previsoes_raw : 323
Tickers com singular matrix : 29
Tickers em df_previsoes     : 294


# Diagnóstico: Comparação de Estratégias de Filtro

São testadas quatro estratégias e comparados: número de tickers rodados,
tickers com erro (singular matrix), tickers na comparação final, QLIKE e MCS.

| # | Estratégia | Filtro zeros | Remove singular matrix |
|---|------------|-------------|------------------------|
| 1 | Singular matrix removido (sem filtro zeros) | nenhum | sim |
| 2 | Filtro ≥70% + remove singular matrix | ≥70% | sim |
| 3 | Filtro ≥50% + remove singular matrix | ≥50% | sim |
| 4 | Sem filtro (NaN via `.dropna()`) | nenhum | não |

In [ ]:
# ── Função de perda QLIKE ────────────────────────────────────────────
def qlike(rv, pred):
    mask = (rv > 0) & (pred > 0)
    r    = np.where(mask, rv / pred, np.nan)
    return np.where(mask, r - np.log(r) - 1, np.nan)


# ── Função de comparação por cenário ────────────────────────────────────
def comparar_cenario(df_prev, threshold_zeros=None, remove_singular=True, label=""):
    """
    threshold_zeros : float | None
        None  -> sem filtro de zeros
        0.70  -> remove tickers com >= 70% de zeros no SVI
        0.50  -> remove tickers com >= 50% de zeros no SVI
        remove_singular : bool
        True  -> remove explicitamente tickers com singular matrix
        False -> mantém; NaN são descartados pelo .dropna() no MCS
    """
    df_sub = df_prev.copy()
    n_inicial = df_sub["ticker_b3"].nunique()

    # Tickers com qualquer erro em rv_pred_m2
    tickers_com_erro = df_sub[df_sub["rv_pred_m2"].isna()]["ticker_b3"].unique()
    n_erro = len(tickers_com_erro)

    # Aplica filtro de zeros
    if threshold_zeros is not None:
        tickers_zeros_ok = zeros_pct[zeros_pct < threshold_zeros].index
        df_sub = df_sub[df_sub["ticker_b3"].isin(tickers_zeros_ok)]
    n_apos_zeros = df_sub["ticker_b3"].nunique()
    n_erro_restante = df_sub[df_sub["rv_pred_m2"].isna()]["ticker_b3"].nunique()

    # Remove singular matrix explicitamente (se solicitado)
    if remove_singular:
        err_sub = df_sub[df_sub["rv_pred_m2"].isna()]["ticker_b3"].unique()
        df_sub = df_sub[~df_sub["ticker_b3"].isin(err_sub)]

    n_comparacao = df_sub["ticker_b3"].nunique()

    # QLIKE por observação
    df_oos = df_sub.query("split == 'out-of-sample'").copy()
    df_oos["qlike_m0"] = qlike(df_oos["rv"], df_oos["rv_pred_m0"])
    df_oos["qlike_m1"] = qlike(df_oos["rv"], df_oos["rv_pred_m1"])
    df_oos["qlike_m2"] = qlike(df_oos["rv"], df_oos["rv_pred_m2"])

    oos_losses = (
        df_oos[list(NOMES)]
        .dropna()          # garante mesmas observações para todos os modelos
        .rename(columns=NOMES)
        .reset_index(drop=True)
    )
    n_obs = len(oos_losses)
    n_obs_descartadas = len(df_oos) - n_obs

    mcs_res = MCS(oos_losses, size=0.10, block_size=5, reps=1000, seed=42)
    mcs_res.compute()
    df_mcs_res = (
        mcs_res.pvalues
        .rename(columns={"Pvalue": "p_valor"})
        .assign(no_conjunto=lambda d: d["p_valor"] >= 0.10)
        .sort_values("p_valor", ascending=False)
    )

    sep = "═" * 60
    print(f"\n{sep}")
    print(f"  {label}")
    print(sep)
    print(f"  Tickers                             : {n_inicial}")
    print(f"  Com erro (singular matrix)          : {n_erro}")
    if threshold_zeros is not None:
        print(f"  Após filtro zeros ≥{threshold_zeros:.0%}             : {n_apos_zeros}  ({n_erro_restante} com erro)")
    print(f"  Na comparação final                  : {n_comparacao}")
    print(f"  Observações OOS usadas / descartadas : {n_obs} / {n_obs_descartadas}")
    print(f"\n  QLIKE médio:")
    for modelo, val in oos_losses.mean().items():
        print(f"    {modelo:<15}: {val:.6f}")
    print(f"\n  MCS (α=10%):")
    print(df_mcs_res.to_string())
    print(f"\n  Conjunto de confiança : {list(mcs_res.included)}")
    print(f"  Excluídos             : {list(mcs_res.excluded)}")

    return df_mcs_res, oos_losses.mean()


In [19]:
df_mcs_1, qlike_1 = comparar_cenario(
    df_previsoes_raw,
    threshold_zeros=None, remove_singular=True,
    label="1. Singular matrix removido (sem filtro zeros)"
)

df_mcs_2, qlike_2 = comparar_cenario(
    df_previsoes_raw,
    threshold_zeros=0.70, remove_singular=True,
    label="2. Filtro ≥70% + singular matrix removido"
)

df_mcs_3, qlike_3 = comparar_cenario(
    df_previsoes_raw,
    threshold_zeros=0.50, remove_singular=True,
    label="3. Filtro ≥50% + singular matrix removido"
)

df_mcs_4, qlike_4 = comparar_cenario(
    df_previsoes_raw,
    threshold_zeros=None, remove_singular=False,
    label="4. Sem filtro (Remove NaN para testes)"
)


════════════════════════════════════════════════════════════
  1. Singular matrix removido (sem filtro zeros)
════════════════════════════════════════════════════════════
  Tickers em df_previsoes            : 323
  Com erro (singular matrix)          : 29
  Na comparação final                  : 294
  Observações OOS usadas / descartadas : 29070 / 1212

  QLIKE médio:
    M0 (RW)        : 38.981838
    M1 (GARCH)     : 0.794214
    M2 (GARCH-X)   : 0.798090

  MCS (α=10%):
              p_valor  no_conjunto
Model name                        
M1 (GARCH)      1.000         True
M2 (GARCH-X)    0.342         True
M0 (RW)         0.008        False

  Conjunto de confiança : ['M1 (GARCH)', 'M2 (GARCH-X)']
  Excluídos             : ['M0 (RW)']

════════════════════════════════════════════════════════════
  2. Filtro ≥70% + singular matrix removido
════════════════════════════════════════════════════════════
  Tickers em df_previsoes            : 323
  Com erro (singular matrix)          :

#  Resultado

In [ ]:
# ── NOTA: qlike() definida na célula de diagnóstico acima

df_oos = df_previsoes.query("split == 'out-of-sample'").copy()
df_oos["qlike_m0"] = qlike(df_oos["rv"], df_oos["rv_pred_m0"])
df_oos["qlike_m1"] = qlike(df_oos["rv"], df_oos["rv_pred_m1"])
df_oos["qlike_m2"] = qlike(df_oos["rv"], df_oos["rv_pred_m2"])

# ── QLIKE médio por ticker — apenas descritivo ─────────────────────
df_qlike = (
    df_oos
    .groupby("ticker_b3")[list(NOMES)]
    .mean()
    .rename(columns=NOMES)
)

# ── Model Confidence Set — teste oficial de significância ──────────
# Perdas pooladas: todas as observações OOS de todos os tickers
oos_losses = (
    df_oos[list(NOMES)]
    .dropna()
    .rename(columns=NOMES)
    .reset_index(drop=True)
)

mcs = MCS(oos_losses, size=0.10, block_size=5, reps=1000, seed=42)
mcs.compute()

df_mcs = (
    mcs.pvalues
    .rename(columns={"Pvalue": "p_valor"})
    .assign(no_conjunto=lambda d: d["p_valor"] >= 0.10)
    .sort_values("p_valor", ascending=False)
)

# ── Salva ──────────────────────────────────────────────────────────
df_qlike.to_csv(CAMINHO + "qlike_por_ticker.csv")
df_mcs.to_csv(CAMINHO + "mcs_resultado.csv")

print(f"\n✅ {len(df_previsoes)} linhas | "
      f"{(df_previsoes['split'] == 'out-of-sample').sum()} OOS")

# ── Resumo γ ───────────────────────────────────────────────────────
n_sig = (df_resumo_params["pct_signif_5pct"] >= 50).sum()
print(f"\n📊 Resumo γ (ΔSVI):")
print(f"   {n_sig}/{len(df_resumo_params)} tickers com γ significativo em ≥50% das janelas OOS")
print(f"\n   Top 5 por % de janelas significativas:")
print(df_resumo_params["pct_signif_5pct"].sort_values(ascending=False).head().to_string())

# ── QLIKE médio global (descritivo) ───────────────────────────────
print(f"\n📊 QLIKE médio por modelo (descritivo):")
print(df_qlike.mean().to_string())

# ── MCS — resultado oficial ────────────────────────────────────────
print(f"\n📊 Model Confidence Set — α = 10%:")
print(df_mcs.to_string())
print(f"\n✅ Conjunto de confiança: {list(mcs.included)}")
print(f"❌ Excluídos:             {list(mcs.excluded)}")


✅ 30282 linhas | 30282 OOS

📊 Resumo γ (ΔSVI):
   65/323 tickers com γ significativo em ≥50% das janelas OOS

   Top 5 por % de janelas significativas:
43     100.0
38     100.0
125    100.0
215    100.0
203    100.0

📊 QLIKE médio por modelo (descritivo):
M0 (RW)         45.881940
M1 (GARCH)       0.947140
M2 (GARCH-X)     1.118282

📊 Model Confidence Set — α = 10%:
              p_valor  no_conjunto
Model name                        
M1 (GARCH)      1.000         True
M2 (GARCH-X)    0.342         True
M0 (RW)         0.008        False

✅ Conjunto de confiança: ['M1 (GARCH)', 'M2 (GARCH-X)']
❌ Excluídos:             ['M0 (RW)']


# Análise exploratória dos resultados

In [ ]:
df_qlike["melhor"] = df_qlike[list(NOMES.values())].idxmin(axis=1)
print(df_qlike["melhor"].value_counts())

# Cruzar γ-significância com qual modelo vence por ticker
tickers_signif = df_resumo_params[df_resumo_params["pct_signif_5pct"] >= 50].index
# Filter tickers_signif to only include those present in df_qlike's index
tickers_signif = tickers_signif.intersection(df_qlike.index)
print(df_qlike.loc[tickers_signif, "melhor"].value_counts())

In [ ]:
# ── Grupos ─────────────────────────────────────────────────────
tickers_m2_vence = df_qlike[df_qlike["melhor"] == "M2 (GARCH-X)"].index
tickers_m2_perde = df_qlike[df_qlike["melhor"] != "M2 (GARCH-X)"].index
tickers_todos    = df_qlike.index

# ── Volume financeiro (R$) — fonte correta ─────────────────────
vol = (
    df_acoes
    .assign(ticker_b3=lambda d: d["ticker_yf"].str.replace(".SA", "", regex=False))
    .groupby("ticker_b3")["vol_fin_medio"]
    .first()
)

# ── Zeros no SVI ───────────────────────────────────────────────
zeros_svi = (
    df_final.groupby("ticker_b3")["svi"]
    .apply(lambda x: (x == 0).mean() * 100)
)

# ── Setor via yfinance ─────────────────────────────────────────
print("⏳ Buscando setor...")
setores = {}
for t in tickers_todos:
    try:
        setores[t] = yf.Ticker(t + ".SA").info.get("sector", "N/D")
    except:
        setores[t] = "N/D"

# ── DataFrame base ─────────────────────────────────────────────
df_analise = df_qlike[["melhor"]].copy()
df_analise["grupo"]        = np.where(df_analise.index.isin(tickers_m2_vence), "M2 vence", "M2 perde")
df_analise["zeros_pct"]    = zeros_svi
df_analise["vol_fin_medio"] = vol
df_analise["setor"]        = pd.Series(setores)

df_analise["zeros_bucket"] = pd.cut(
    df_analise["zeros_pct"],
    bins=range(0, 110, 10), right=False,
    labels=[f"{i}-{i+10}%" for i in range(0, 100, 10)]
)
df_analise["vol_bucket"] = pd.cut(
    df_analise["vol_fin_medio"],
    bins=[0, 1e6, 10e6, 100e6, 500e6, float("inf")],
    labels=["<1M", "1-10M", "10-100M", "100-500M", ">500M"]
)

# ── Função para montar tabela comparativa ──────────────────────
def tabela_comparativa(coluna, titulo):
    tab = (
        df_analise.groupby([coluna, "grupo"])
        .size()
        .unstack(fill_value=0)
        .reindex(columns=["M2 perde", "M2 vence"], fill_value=0)
    )
    tab["Total"] = tab.sum(axis=1)
    tab.index.name = "Métrica"
    tab.columns.name = None
    print(f"\n{'═'*55}")
    print(f"  {titulo}")
    print(f"{'═'*55}")
    print(tab.to_string())

# ── Tabelas ────────────────────────────────────────────────────
tabela_comparativa("zeros_bucket", "1. Zeros no SVI")
tabela_comparativa("vol_bucket",   "2. Volume financeiro médio (R$)")
tabela_comparativa("setor",        "3. Setor")

# ── Mediana do volume por grupo ────────────────────────────────
print("\n  Mediana volume financeiro por grupo:")
print(
    df_analise.groupby("grupo")["vol_fin_medio"]
    .median()
    .apply(lambda x: f"R$ {x:,.0f}")
    .to_string()
)

In [ ]:
print(df_analise[df_analise["vol_bucket"] == ">500M"][["grupo", "vol_fin_medio"]].sort_values("vol_fin_medio", ascending=False))
print(df_analise.groupby("grupo")["zeros_pct"].median())

In [ ]:
# Verificar PDGR3 no df_acoes
print(df_acoes[df_acoes["ticker_yf"] == "PDGR3.SA"])

# E no df_final para ver o volume semanal
print(df_final[df_final["ticker_b3"] == "PDGR3"][["semana", "volume_total", "preco_fechamento"]].describe())

In [ ]:
import matplotlib.pyplot as plt

# % de tickers com svi=0 por semana, separado por grupo
df_final["grupo"] = np.where(
    df_final["ticker_b3"].isin(tickers_m2_vence), "M2 vence", "M2 perde"
)

zeros_semana = (
    df_final.groupby(["semana", "grupo"])
    .apply(lambda x: (x["svi"] == 0).mean() * 100)
    .reset_index(name="pct_zeros")
)

fig, ax = plt.subplots(figsize=(14, 5))

for grupo, cor in [("M2 vence", "#1f77b4"), ("M2 perde", "#d62728")]:
    d = zeros_semana[zeros_semana["grupo"] == grupo]
    ax.plot(d["semana"], d["pct_zeros"], label=grupo, color=cor, linewidth=1.5)

ax.set_title("% de tickers com SVI = 0 por semana", fontsize=13)
ax.set_xlabel("Semana")
ax.set_ylabel("% de tickers com SVI = 0")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()